# Feature engineering et encodage des variables catégorielles
### Scoring de risque de crédit — Séance 5

On repart du fichier nettoyé en S4 (`data/processed/Loan_default_clean.csv`) — mais comme ce fichier n'est pas encore recréé dans cet environnement, on refait d'abord rapidement les 2 étapes de S4 (chargement + indicateur d'incohérence Age/MonthsEmployed) avant d'attaquer le vrai sujet du jour : rendre les données exploitables par un modèle.

## Étape 0 — Recharger le point de départ (résultat de S4)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Loan_default.csv")

# On refait le calcul de l'indicateur de coherence Age / MonthsEmployed (vu en S4)
mois_emploi_maximum_plausible = (df["Age"] - 16) * 12
df["AgeEmploymentIncoherent"] = (df["MonthsEmployed"] > mois_emploi_maximum_plausible).astype(int)

df.shape

(255347, 19)

---
# Pourquoi encoder les variables catégorielles ?

Un modèle de Machine Learning (régression logistique, arbre de décision...) ne travaille qu'avec des **nombres**. Nos 7 variables catégorielles (`Education`, `EmploymentType`, `MaritalStatus`, `HasMortgage`, `HasDependents`, `LoanPurpose`, `HasCoSigner`) sont actuellement du texte — il faut les transformer, sans perdre l'information qu'elles portent.

**Il n'existe pas une seule bonne méthode : le choix dépend de la nature de chaque variable.**

## 1. Les variables binaires (Oui/Non) → 0/1 direct

`HasMortgage`, `HasDependents`, `HasCoSigner` n'ont que 2 valeurs possibles (Yes/No). Pas besoin d'une méthode compliquée : on remplace directement Yes par 1 et No par 0.

In [2]:
colonnes_binaires = ["HasMortgage", "HasDependents", "HasCoSigner"]

for colonne in colonnes_binaires:
    df[colonne] = df[colonne].map({"Yes": 1, "No": 0})

df[colonnes_binaires].head()

,HasMortgage,HasDependents,HasCoSigner
0,1,1,1
1,0,0,1
2,1,1,0
3,0,0,0
4,0,1,0


## 2. Une variable avec un ordre naturel → encodage ordinal

`Education` a un ordre logique : High School < Bachelor's < Master's < PhD (chaque niveau représente "plus" d'études que le précédent). On encode donc avec des nombres qui respectent cet ordre — contrairement à un One-Hot Encoding qui traiterait ces 4 niveaux comme complètement indépendants et perdrait cette notion de hiérarchie.

In [3]:
ordre_education = {
    "High School": 0,
    "Bachelor's": 1,
    "Master's": 2,
    "PhD": 3,
}

df["Education_encoded"] = df["Education"].map(ordre_education)

df[["Education", "Education_encoded"]].drop_duplicates().sort_values("Education_encoded")

,Education,Education_encoded
3,High School,0
0,Bachelor's,1
1,Master's,2
7,PhD,3


## 3. Les variables sans ordre naturel → One-Hot Encoding

`EmploymentType`, `MaritalStatus` et `LoanPurpose` n'ont **pas** d'ordre logique entre leurs catégories (rien ne dit que "Married" est "plus" que "Single"). Leur donner un simple numéro (0, 1, 2...) tromperait le modèle en lui faisant croire à un ordre qui n'existe pas.

**One-Hot Encoding** : on crée une colonne binaire par catégorie. Par exemple, `EmploymentType` (4 catégories) devient 4 colonnes : `EmploymentType_Full-time`, `EmploymentType_Part-time`, etc., chacune valant 1 si la ligne correspond à cette catégorie, 0 sinon.

In [4]:
colonnes_a_one_hot = ["EmploymentType", "MaritalStatus", "LoanPurpose"]

# pd.get_dummies fait exactement ca : une colonne binaire par categorie
df_encode = pd.get_dummies(df, columns=colonnes_a_one_hot, prefix=colonnes_a_one_hot)

# On verifie que de nouvelles colonnes sont bien apparues
nouvelles_colonnes = [c for c in df_encode.columns if any(c.startswith(p + "_") for p in colonnes_a_one_hot)]
print("Nouvelles colonnes creees :", nouvelles_colonnes)
print()
print("Nombre de colonnes avant :", df.shape[1], " -> apres :", df_encode.shape[1])

Nouvelles colonnes creees : ['EmploymentType_Full-time', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Divorced', 'MaritalStatus_Married', 'MaritalStatus_Single', 'LoanPurpose_Auto', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other']

Nombre de colonnes avant : 20  -> apres : 29


**Remarque technique :** avec 4 catégories dans `EmploymentType`, on obtient 4 nouvelles colonnes. Certains praticiens suppriment une des colonnes (`drop_first=True`) pour éviter une redondance parfaite entre elles (dite "colinéarité") — utile en régression logistique classique, moins critique pour les méthodes d'ensemble (Random Forest, Gradient Boosting) qu'on comparera en S6. On garde ici toutes les colonnes pour rester transparent et lisible ; ce choix pourra être révisé au moment de comparer les modèles.

Par défaut, `pd.get_dummies` produit des colonnes de type `bool` (True/False). On les convertit en `int` (1/0) pour que l'ensemble du jeu de données soit uniformément numérique.

In [5]:
colonnes_one_hot_creees = [col for col in df_encode.columns if any(col.startswith(p + "_") for p in colonnes_a_one_hot)]

for colonne in colonnes_one_hot_creees:
    df_encode[colonne] = df_encode[colonne].astype(int)

df_encode[colonnes_one_hot_creees].dtypes

EmploymentType_Full-time        int64
EmploymentType_Part-time        int64
EmploymentType_Self-employed    int64
EmploymentType_Unemployed       int64
MaritalStatus_Divorced          int64
MaritalStatus_Married           int64
MaritalStatus_Single            int64
LoanPurpose_Auto                int64
LoanPurpose_Business            int64
LoanPurpose_Education           int64
LoanPurpose_Home                int64
LoanPurpose_Other               int64
dtype: object

## 4. Feature engineering — créer une nouvelle variable utile

Le feature engineering va plus loin que l'encodage : il s'agit de **construire de nouvelles variables** à partir des existantes, quand elles peuvent apporter une information que le modèle ne verrait pas directement.

On crée ici un ratio **prêt / revenu** (`LoanToIncomeRatio`) : pour un même montant de prêt, le risque n'est pas le même selon que l'emprunteur gagne 20 000 ou 140 000. Ce ratio résume cette relation en une seule variable, alors que `LoanAmount` et `Income` pris séparément ne la capturent pas directement.

In [6]:
df_encode["LoanToIncomeRatio"] = df_encode["LoanAmount"] / df_encode["Income"]

df_encode[["LoanAmount", "Income", "LoanToIncomeRatio"]].head()

,LoanAmount,Income,LoanToIncomeRatio
0,50587,85994,0.588262
1,124440,50432,2.467481
2,129188,84208,1.534154
3,44799,31713,1.412638
4,9139,20437,0.447179


In [7]:
# Petite verification de bon sens : ce ratio a-t-il un lien avec le risque de defaut ?
from scipy import stats

groupe_non_defaut = df_encode[df_encode["Default"] == 0]["LoanToIncomeRatio"]
groupe_defaut = df_encode[df_encode["Default"] == 1]["LoanToIncomeRatio"]

t_stat, p_value = stats.ttest_ind(groupe_non_defaut, groupe_defaut)
print(f"p-value = {p_value:.4f}")
print("Significatif" if p_value < 0.05 else "Non significatif")

p-value = 0.0000
Significatif


## 5. Nettoyer les colonnes qui ne servent plus

- `LoanID` : identifiant, pas une caractéristique prédictive — on le retire des features mais on le garde à part (utile pour retrouver une ligne précise si besoin).
- `Education` (texte d'origine) : remplacée par `Education_encoded`, on peut retirer la version texte.
- Les colonnes déjà transformées par le One-Hot Encoding (`EmploymentType`, `MaritalStatus`, `LoanPurpose`) ont déjà été retirées automatiquement par `pd.get_dummies`.

In [8]:
identifiants = df_encode["LoanID"]

df_final = df_encode.drop(columns=["LoanID", "Education"])

print("Colonnes finales :", df_final.shape[1])
df_final.dtypes

Colonnes finales : 28


Age                               int64
Income                            int64
LoanAmount                        int64
CreditScore                       int64
MonthsEmployed                    int64
NumCreditLines                    int64
InterestRate                    float64
LoanTerm                          int64
DTIRatio                        float64
HasMortgage                       int64
HasDependents                     int64
HasCoSigner                       int64
Default                           int64
AgeEmploymentIncoherent           int64
Education_encoded                 int64
EmploymentType_Full-time          int64
EmploymentType_Part-time          int64
EmploymentType_Self-employed      int64
EmploymentType_Unemployed         int64
MaritalStatus_Divorced            int64
MaritalStatus_Married             int64
MaritalStatus_Single              int64
LoanPurpose_Auto                  int64
LoanPurpose_Business              int64
LoanPurpose_Education             int64


## 6. Sauvegarder le jeu de données encodé

In [9]:
df_final.to_csv("../data/processed/Loan_default_features.csv", index=False)
print("Fichier sauvegardé :", df_final.shape)

Fichier sauvegardé : (255347, 28)


## Résumé

| Type de variable | Méthode utilisée | Exemple |
|---|---|---|
| Binaire (Yes/No) | Remplacement direct 0/1 | HasMortgage, HasDependents, HasCoSigner |
| Catégorielle avec ordre | Encodage ordinal | Education (0 à 3) |
| Catégorielle sans ordre | One-Hot Encoding | EmploymentType, MaritalStatus, LoanPurpose |
| Nouvelle variable | Feature engineering | LoanToIncomeRatio = LoanAmount / Income |

Le jeu de données est maintenant **entièrement numérique**, prêt pour la modélisation (S6). Prochaine étape : séparer les données en jeu d'entraînement / jeu de test, puis entraîner un premier modèle de référence (régression logistique).